In [1]:
# set wd
import os
from pathlib import Path
print(os.getcwd())
wd = Path(os.getcwd())
if wd.name == "notebooks":
    %cd ..
print(f"Working Dir Base: {(os.getcwd())}")

/Users/peli/Projects/Repositories/MEGPypes/notebooks
/Users/peli/Projects/Repositories/MEGPypes
Working Dir Base: /Users/peli/Projects/Repositories/MEGPypes


In [2]:
import yaml
import time
from bids.layout import BIDSLayout
# import
from megpypes.pipelines.meg_preprocessing import create_meg_preprocessing


/Users/peli/Projects/Repositories/MEGPypes/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loaded a MEG dataset from here:
https://openneuro.org/datasets/ds006629/versions/1.0.1

using datalad
```
datalad install -s https://github.com/OpenNeuroDatasets/ds006629.git data/ds006629
```
```
cd output/ds006629
datalad get -r .
```


First we need to grab data from our dataset.
We assume the data operate within a bids compliant data structure
https://peerherholz.github.io/workshop_weizmann/nipype/notebooks/basic_data_input_bids.html



In [3]:
# do some data inspection using pybids?
layout = BIDSLayout("data/ds006629/")
print(layout)
# subjects
subjects = layout.get_subjects()
print(f"Subjects: {subjects}")
# sessions
sessions = layout.get_sessions()
print(f"Sessions: {sessions}")
# task
layout.get_tasks()
# datatypes
bidstypes = layout.get_datatypes()
print(f"Data types: {bidstypes}")
# suffixes
print(layout.get_suffixes(datatype='func'))
# see tasks
layout.get_tasks()
print(f"Tasks: {layout.get_tasks()}")
# see dataset description
layout.get_dataset_description()
#

# see data metadata


BIDS Layout: ...itories/MEGPypes/data/ds006629 | Subjects: 19 | Sessions: 0 | Runs: 19
Subjects: ['01', '02', '04', '05', '06', '07', '08', '09', '10', '11', '12', '14', '15', '16', '17', '18', '19', '20', '21']
Sessions: []
Data types: ['meg']
[]
Tasks: ['MMNHCS', 'noise']


{'Name': 'SINGSING',
 'BIDSVersion': '1.7.0',
 'License': 'CC0',
 'DatasetType': 'raw',
 'Authors': ['Valerie Chanoine',
  'Jean-Michel Badier',
  'Mireille Besson',
  'Talya Inbar'],
 'Acknowledgements': 'MEG data acquisition was performed in the MEG Centre (Timone Hospital, Marseille, France)',
 'Funding': ['This research has been supported by funding from the Institute of Convergence ILCB (France 2030, ANR-16-CONV-0002) and the Excellence Initiative of Aix-Marseille University A*MIDEX (ANR-11-IDEX-0001-02)'],
 'ReferencesAndLinks': ['a data paper',
  'a resource to be cited when using the data'],
 'DatasetDOI': 'doi:10.18112/openneuro.ds006629.v1.0.1',
 'GeneratedBy': [{'Name': 'MNE-BIDS',
   'Version': '0.14',
   'Description': 'MNE-BIDS is a Python package that allows you to read and write BIDS-compatible datasets with the help of MNE-Python.'}],
 'SourceDatasets': [{'DOI': 'doi:10.18112/openneuro.ds006629.v1.0.0',
   'URL': 'https://openneuro.org/datasets/ds006629',
   'Version':

In [4]:
# Load configs
import os
import yaml
from nipype import config as nconfig

config_path = "config/config_effort.yaml"
with open(config_path, "r") as yamlfile:
    config = yaml.load(yamlfile, Loader=yaml.FullLoader)

wf_config = config['workflow']
paths_config = config['paths']

# Configure Nipype logging (applies to all subprocesses)
nconfig.update_config({
    'logging': {
        'log_directory': os.path.join(paths_config["workdir"], 'logs'),
        'log_to_file': True,
        'interface_level': 'info',
        'workflow_level': 'info',
    },
    'execution': {
        'crashdump_dir': os.path.abspath('crashes'),
        'remove_unnecessary_outputs': False,
    }
})

# Create workflow
wf = create_meg_preprocessing(
    basedir=paths_config['basedir'], 
    workdir=paths_config['workdir'], 
    output_dir=paths_config['outputdir'],
    file_templates=paths_config.get('file_templates', None),
    iterable_fields=paths_config.get('iterable_fields', None),
    iterable_values=paths_config.get('iterable_values', None),
    pipeline_config=config['pipeline_config']
    )

# visualize workflow graph
wf.write_graph(graph2use='colored', simple_form=True)
print(f"Workflow graph saved to: {wf.base_dir}/megpreproc/graph.png")

# Run workflow
n_workers = wf_config.get("n_workers", max(1, os.cpu_count() - 2))
print(f"Running with {n_workers} workers")

result = wf.run(
    plugin=wf_config["plugin"],
    plugin_args={"n_procs": n_workers}
)

raw_dir /Users/peli/Projects/Repositories/MEGPypes/data/effort
Iterating over fields: ['subject', 'session']
Provided iterable values: {'subject': ['0001', '0002'], 'session': ['01', '02']}
Processing iterable field: subject
Using custom values for field: subject
Processing iterable field: session
Using custom values for field: session
Final iterables dict: {'subject': ['0001', '0002'], 'session': ['01', '02']}
Valid inputs for initial_preproc: {'stim_channel', 'max_buffer', 'in_file', 'min_buffer', 'h_freq', 'gradcomp_order', 'out_file', 'trait_modified', 'trait_added', 'l_freq', 'gradcomp_auto'}
Step 'crop' argument 'stim_channel': None
Step 'crop' argument 'min_buffer': -0.2
Step 'crop' argument 'max_buffer': 0.5
Step 'filter' argument 'l_freq': 1.0
Step 'filter' argument 'h_freq': 100
Step 'gradcomp' argument 'gradcomp_auto': True
Step 'gradcomp' argument 'order': None
Valid inputs for artifact_rejection: {'enable_zapline', 'out_file', 'ica_random_state', 'ica_n_components', 'enabl

2026-03-12 17:39:02,754 [INFO] megpypes.interfaces.epoching: NODE: Epoching | In-File: /Users/peli/Projects/Repositories/MEGPypes/workdir/megpreproc/_session_01_subject_0001/artifact_rejection/artifact_cleaned_raw.fif
2026-03-12 17:39:02,754 [INFO] megpypes.interfaces.epoching: Epoching for event: cue_onset
2026-03-12 17:39:02,754 [INFO] megpypes.interfaces.epoching: id: 19, tmin: -1.0, tmax: 1.0


Opening raw data file /Users/peli/Projects/Repositories/MEGPypes/workdir/megpreproc/_session_01_subject_0001/artifact_rejection/artifact_cleaned_raw.fif...
    Read 5 compensation matrices
    Range : 6996 ... 773425 =      5.970 ...   659.989 secs
Ready.
Current compensation grade : 0
Reading 0 ... 766429  =      0.000 ...   654.019 secs...
Finding events on: HTRG001, HTRG002, UDIO001
Trigger channel HTRG001 has a non-zero initial value of 85 (consider using initial_event=True to detect this event)
Trigger channel HTRG002 has a non-zero initial value of 1 (consider using initial_event=True to detect this event)
589 events found on stim channel UDIO001
Event IDs: [   1    2    4    6    7    8   10   13   16   19   22   30   50   55
  135  141  147  178 1024 2048 2062 2065 2187 4096 4224]


Default mne.find_events() failed. Attempting fallback.


260312-17:39:03,647 nipype.workflow INFO:
	 [Node] Finished "epoching", elapsed time 0.89089s.
260312-17:39:03,647 nipype.workflow WARNING:
	 Storing result file without outputs
260312-17:39:03,648 nipype.workflow WARNING:
	 [Node] Error on "megpreproc.epoching" (/Users/peli/Projects/Repositories/MEGPypes/workdir/megpreproc/_session_01_subject_0001/_event_id_19_event_label_cue_onset_event_tmax_1.0_event_tmin_-1.0/epoching)
260312-17:39:03,650 nipype.workflow ERROR:
	 Node epoching.aI.a0.b0 failed to run on host Elisiuss-MacBook-Pro.local.
260312-17:39:03,651 nipype.workflow ERROR:
	 Saving crash info to /Users/peli/Projects/Repositories/MEGPypes/crashes/crash-20260312-173903-peli-epoching.aI.a0.b0-2b2bb84b-6344-4526-a44f-5a334d5f06c1.pklz
Traceback (most recent call last):
  File "/Users/peli/Projects/Repositories/MEGPypes/.venv/lib/python3.13/site-packages/nipype/pipeline/plugins/linear.py", line 45, in run
    node.run(updatehash=updatehash)
    ~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File

...

2026-03-12 17:41:46,567 [INFO] megpypes.interfaces.epoching: Running autoreject on epochs
2026-03-12 17:41:46,570 [INFO] megpypes.interfaces.epoching: AutoReject picks: Counter({'mag': 270})


Removing 5 compensators from info because not all compensation channels were picked.
Running autoreject on ch_type=mag


100%|██████████| Creating augmented epochs : 270/270 [00:10<00:00,   26.40it/s]
100%|██████████| Computing thresholds ... : 270/270 [00:39<00:00,    6.79it/s]




















100%|██████████| Repairing epochs : 39/39 [00:00<00:00,   73.33it/s]


































100%|██████████| Repairing epochs : 39/39 [00:01<00:00,   26.98it/s]






















100%|██████████| Fold : 10/10 [00:01<00:00,    7.17it/s]

































100%|██████████| Repairing epochs : 39/39 [00:01<00:00,   26.95it/s]






















100%|██████████| Fold : 10/10 [00:01<00:00,    7.38it/s]


































100%|██████████| Repairing epochs : 39/39 [00:01<00:00,   27.84it/s]






















100%|██████████| Fold : 10/10 [00:01<00:00,    7.38it/s]
100%|██████████| n_interp : 3/3 [00:09<00:00,    3.02s/it]





Estimated consensus=0.50 and n_interpolate=32
260312-17:42:50,397 nipype.workflow INFO:
	 [Node] Finished "epoching", elapsed time 65.062s.
260312-17:42:50,397 nipype.workflow WARNING:
	 Storing result file without outputs
260312-17:42:50,398 nipype.workflow WARNING:
	 [Node] Error on "megpreproc.epoching" (/Users/peli/Projects/Repositories/MEGPypes/workdir/megpreproc/_session_02_subject_0002/_event_id_8_event_label_feedback_onset_event_tmax_1.5_event_tmin_-1.0/epoching)
260312-17:42:50,399 nipype.workflow ERROR:
	 Node epoching.aI.a2.b1 failed to run on host Elisiuss-MacBook-Pro.local.
260312-17:42:50,399 nipype.workflow ERROR:
	 Saving crash info to /Users/peli/Projects/Repositories/MEGPypes/crashes/crash-20260312-174250-peli-epoching.aI.a2.b1-48196619-dea4-4735-a440-4d4238ec1c59.pklz
Traceback (most recent call last):
  File "/Users/peli/Projects/Repositories/MEGPypes/.venv/lib/python3.13/site-packages/nipype/pipeline/plugins/linear.py", line 45, in run
    node.run(updatehash=u

RuntimeError: 6 raised. Re-raising first.